<a href="https://colab.research.google.com/github/TAMIDSpiyalong/Advanced-Concepts-in-Machine-Learning-for-Energy/blob/main/Lecture_4d_GPT2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#The Transformer Architecture

## 1. INTRODUCTION

The Transformer architecture was introduced in the paper "Attention Is All You Need" (Vaswani et al., 2017).
We'll examine the main components of a (encoder-style) Transformer layer step-by-step:

 - Token Embeddings
 - Positional Encoding
 - Multi-Head Self-Attention
 - Feed-Forward Network
 - Residual Connections & Layer Normalization
 - Putting them together into a Transformer Block
 - Finally, building a multi-layer Transformer Encoder
 - Show how GPT-2 (a decoder-only Transformer) can be loaded from Hugging Face

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import math

# If you're in a fresh environment and need transformers:
# # !pip install transformers

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

## 2. KEY COMPONENTS OF A TRANSFORMER

### 2.1 Token Embeddings

A simple embedding layer that converts token IDs (integers) into vectors of dimension `d_model`.

**Demo**: We'll:
 1. Create a random batch of token IDs (batch_size=2, seq_length=5).
 2. Pass them through the embedding layer.
 3. Print the shape of the output.

In [ ]:
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super(TokenEmbedding, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x):
        # x shape: (batch_size, seq_length)
        # output shape: (batch_size, seq_length, d_model)
        return self.embedding(x)

print("\n-- Demo: TokenEmbedding --")
vocab_size = 100
d_model = 8
sample_batch = torch.randint(0, vocab_size, (2, 5))  # (batch_size=2, seq_length=5)

embedding_layer = TokenEmbedding(vocab_size, d_model)
embedded_output = embedding_layer(sample_batch)
print("Input shape:", sample_batch.shape)
print("Output shape (embedded):", embedded_output.shape)
print("-- End of Demo --")


-- Demo: TokenEmbedding --
Input shape: torch.Size([2, 5])
Output shape (embedded): torch.Size([2, 5, 8])
-- End of Demo --


### 2.2 Positional Encoding

The Transformer doesn't inherently understand the ordering of tokens. We add a positional encoding
to each token embedding so that the model knows the relative/absolute positions.

We'll implement the sinusoidal approach from Vaswani et al.:


PE(pos, 2i) = \sin\bigl(\frac{pos}{10000^{2i/d_{\text{model}}}}\bigr), \quad
PE(pos, 2i+1) = \cos\bigl(\frac{pos}{10000^{2i/d_{\text{model}}}}\bigr)


Demo: We'll:
 1. Use the output of `TokenEmbedding` from the previous step.
 2. Add positional encodings to it.
 3. Print shape and show the effect on a small slice.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)  # shape: (max_len, 1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # Register pe as a buffer so it won't be trained
        self.register_buffer('pe', pe.unsqueeze(0))  # shape: (1, max_len, d_model)

    def forward(self, x):
        # x: (batch_size, seq_length, d_model)
        seq_length = x.size(1)
        # Add positional encoding up to seq_length
        x = x + self.pe[:, :seq_length, :].to(x.device)
        return x


print("\n-- Demo: PositionalEncoding --")
pos_enc_layer = PositionalEncoding(d_model, max_len=10)  # just a small max_len for demonstration
with_pe = pos_enc_layer(embedded_output)  # from previous embedding demo
print("Input shape (embedded_output):", embedded_output.shape)
print("Output shape (pos-encoded):", with_pe.shape)
print("First example, first token embedding (before PE):\n", embedded_output[0,0,:])
print("First example, first token embedding (after PE):\n", with_pe[0,0,:])
print("-- End of Demo --")


-- Demo: PositionalEncoding --
Input shape (embedded_output): torch.Size([2, 5, 8])
Output shape (pos-encoded): torch.Size([2, 5, 8])
First example, first token embedding (before PE):
 tensor([ 2.8520, -0.7436,  0.1954, -1.3350,  0.3945,  1.7060, -0.7939,  0.3752],
       grad_fn=<SelectBackward0>)
First example, first token embedding (after PE):
 tensor([ 2.8520,  0.2564,  0.1954, -0.3350,  0.3945,  2.7060, -0.7939,  1.3752],
       grad_fn=<SelectBackward0>)
-- End of Demo --


### 2.3 Multi-Head Self-Attention

Self-attention: each position can attend to every position (including itself) with a learned weighting.

**Demo**:
 1. We'll create a `MultiHeadSelfAttention` object with `num_heads=2`.
 2. Pass a small batch of embeddings (pos-encoded) through it.
 3. Print the output shape.

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadSelfAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        self.d_k = d_model // num_heads

        self.query = nn.Linear(d_model, d_model)
        self.key = nn.Linear(d_model, d_model)
        self.value = nn.Linear(d_model, d_model)

        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        # x: (batch_size, seq_length, d_model)
        batch_size, seq_length, _ = x.size()

        # Linear projections
        Q = self.query(x)  # (batch_size, seq_length, d_model)
        K = self.key(x)
        V = self.value(x)

        # Reshape to (batch_size, num_heads, seq_length, d_k)
        Q = Q.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)

        # Compute attention scores: (batch_size, num_heads, seq_length, seq_length)
        scores = torch.matmul(Q, K.transpose(-1, -2)) / math.sqrt(self.d_k)

        if mask is not None:
            # mask shape is typically (batch_size, 1, seq_length, seq_length) or (batch_size, seq_length)
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = torch.softmax(scores, dim=-1)

        # Weighted sum of values: (batch_size, num_heads, seq_length, d_k)
        attn_output = torch.matmul(attn_weights, V)

        # Transpose back and combine heads
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)

        # Final linear
        out = self.out_proj(attn_output)
        return out


print("\n-- Demo: MultiHeadSelfAttention --")
attention_layer = MultiHeadSelfAttention(d_model=d_model, num_heads=2)
attn_output = attention_layer(with_pe)  # with_pe from the PositionalEncoding demo
print("Input shape (pos-encoded embeddings):", with_pe.shape)
print("Output shape (after self-attention):", attn_output.shape)
print("-- End of Demo --")


-- Demo: MultiHeadSelfAttention --
Input shape (pos-encoded embeddings): torch.Size([2, 5, 8])
Output shape (after self-attention): torch.Size([2, 5, 8])
-- End of Demo --


### 2.4 Feed-Forward Network (Position-wise)

A 2-layer MLP applied to each position independently:
\[
\text{FFN}(x) = \max(0, xW_1 + b_1) W_2 + b_2
\]

**Demo**:
 1. Construct the feed-forward network with `d_ff=16`.
 2. Pass the attention output through it.
 3. Print the shape.

In [ ]:
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionwiseFeedForward, self).__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        x = self.linear1(x)
        x = torch.relu(x)
        x = self.linear2(x)
        return x

print("\n-- Demo: PositionwiseFeedForward --")
ffn_layer = PositionwiseFeedForward(d_model=d_model, d_ff=16)
ffn_output = ffn_layer(attn_output)
print("Input shape (after self-attn):", attn_output.shape)
print("Output shape (after feed-forward):", ffn_output.shape)
print("-- End of Demo --")


-- Demo: PositionwiseFeedForward --
Input shape (after self-attn): torch.Size([2, 5, 8])
Output shape (after feed-forward): torch.Size([2, 5, 8])
-- End of Demo --


### 2.5 Residual & Layer Normalization

Each sub-layer (attention or feed-forward) is wrapped with:
 1. Residual connection
 2. Layer normalization

We'll define a small sublayer wrapper:

**Demo**:
 1. Use the sublayer wrapper to apply self-attention to the input with a residual connection.
 2. Print shape.

In [ ]:
class SublayerConnection(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super(SublayerConnection, self).__init__()
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer):
        # x: (batch_size, seq_length, d_model)
        # sublayer is a function (could be self-attn or feed-forward)
        normed_x = self.norm(x)
        out = sublayer(normed_x)
        return x + self.dropout(out)  # residual connection


print("\n-- Demo: SublayerConnection (with multi-head attention) --")
sublayer = SublayerConnection(d_model=d_model, dropout=0.1)
# We'll define a "temporary" function that calls our attention layer
def attn_sublayer(x_input):
    return attention_layer(x_input)  # reusing attention_layer from above

# We pass the attn_sublayer function in
sublayer_output = sublayer(with_pe, attn_sublayer)
print("Input shape:", with_pe.shape)
print("Output shape (with residual + layernorm):", sublayer_output.shape)
print("-- End of Demo --")


-- Demo: SublayerConnection (with multi-head attention) --
Input shape: torch.Size([2, 5, 8])
Output shape (with residual + layernorm): torch.Size([2, 5, 8])
-- End of Demo --


## 3. PUTTING IT ALL TOGETHER: TRANSFORMER BLOCK

A single Transformer Encoder Block typically has:
 1. Sublayer: Multi-Head Self-Attention
 2. Sublayer: Feed-Forward

Each with residual + layer normalization.

**Demo**:
 1. Build a `TransformerEncoderLayer`.
 2. Pass the positional-encoded embeddings through it.

In [ ]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(TransformerEncoderLayer, self).__init__()
        self.self_attn = MultiHeadSelfAttention(d_model, num_heads)
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff)

        self.sublayer1 = SublayerConnection(d_model, dropout)
        self.sublayer2 = SublayerConnection(d_model, dropout)

    def forward(self, x, mask=None):
        # 1. Self-Attention sublayer
        x = self.sublayer1(x, lambda _x: self.self_attn(_x, mask=mask))
        # 2. Feed-forward sublayer
        x = self.sublayer2(x, self.feed_forward)
        return x

# Demo for TransformerEncoderLayer
if __name__ == "__main__":
    print("\n-- Demo: TransformerEncoderLayer --")
    encoder_layer = TransformerEncoderLayer(d_model=d_model, num_heads=2, d_ff=16, dropout=0.1)
    layer_output = encoder_layer(with_pe)  # with_pe is from earlier demo
    print("Input shape (pos-encoded embeddings):", with_pe.shape)
    print("Output shape (after 1 Transformer block):", layer_output.shape)
    print("-- End of Demo --")


-- Demo: TransformerEncoderLayer --
Input shape (pos-encoded embeddings): torch.Size([2, 5, 8])
Output shape (after 1 Transformer block): torch.Size([2, 5, 8])
-- End of Demo --


## 4. BUILDING A MULTI-LAYER TRANSFORMER ENCODER

We can stack multiple `TransformerEncoderLayer` objects to form a full encoder.
We also include the TokenEmbedding and PositionalEncoding at the start.

**Demo**:
 1. Build a `TransformerEncoder` with 2 layers.
 2. Pass some random token IDs to see the final output.

In [ ]:
class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, max_seq_len=100, dropout=0.1):
        super(TransformerEncoder, self).__init__()
        self.token_embedding = TokenEmbedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len=max_seq_len)

        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        # x: (batch_size, seq_length)
        # 1. Token + Position
        x = self.token_embedding(x)
        x = self.pos_encoding(x)

        # 2. Stacked layers
        for layer in self.layers:
            x = layer(x, mask=mask)

        # 3. Final layer norm
        x = self.norm(x)
        return x


print("\n-- Demo: TransformerEncoder (2 layers) --")
vocab_size = 1000
d_model = 16
num_heads = 2
d_ff = 32
num_layers = 2
max_seq_len = 10

# Instantiate
encoder = TransformerEncoder(vocab_size, d_model, num_heads, d_ff, num_layers, max_seq_len).to(device)

# Dummy input: (batch_size=2, seq_length=10)
dummy_input = torch.randint(0, vocab_size, (2, 10)).to(device)

# Forward pass
encoder_output = encoder(dummy_input)
print("Encoder output shape:", encoder_output.shape)  # should be (2, 10, d_model)
print("-- End of Demo --")


-- Demo: TransformerEncoder (2 layers) --
Encoder output shape: torch.Size([2, 10, 16])
-- End of Demo --


## 5. EXAMPLE: GPT-2 STYLE TRANSFORMER via HUGGING FACE

 GPT-2 is a decoder-only Transformer that uses causal self-attention (no looking ahead in the sequence).
 Here we'll just instantiate a small GPT-2 via the `transformers` library and show a quick generation.

 **Demo**:
  1. Create a small GPT2 config.
  2. Instantiate GPT2LMHeadModel.
  3. Tokenize a prompt and generate text.

 *Note*: For real usage, you'd typically load pretrained weights from "gpt2" or "gpt2-medium".

In [ ]:
print("\n-- Demo: GPT-2 from Hugging Face (small config) --")
try:
    from transformers import GPT2LMHeadModel, GPT2Config, GPT2Tokenizer

    # 1. Create a GPT2 config (small for demo)
    config = GPT2Config(
        vocab_size=50257,  # GPT-2 default
        n_embd=64,         # smaller embedding size for demonstration
        n_layer=2,         # number of Transformer decoder blocks
        n_head=4,          # number of heads
        bos_token_id=50256,
        eos_token_id=50256
    )

    # 2. Instantiate the model
    model = GPT2LMHeadModel(config).to(device)

    # 3. Tokenizer
    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

    # 4. Generate text with the model
    prompt = "Texas"
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

    # Typically use model.generate for text generation
    with torch.no_grad():
        output_sequences = model.generate(
            input_ids=input_ids,
            max_length=30,
            temperature=1.0,
            do_sample=True,
            top_k=50,
            top_p=0.95
        )

    # Decode output
    generated_text = tokenizer.decode(output_sequences[0], skip_special_tokens=True)
    print("Generated text:\n", generated_text)
except ImportError:
    print("Hugging Face 'transformers' not installed. Please install via `pip install transformers`.")
print("-- End of GPT-2 Demo --")


-- Demo: GPT-2 from Hugging Face (small config) --


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Generated text:
 Texasoris 4000 Lands continuing hygiene customizationMoney stabbingOverview proclaiming 171 wavegroupsgroups Janeiro clamahar sulfur Paradise suppression Reyn AG Classificationersen mathematic cybersecurity believesmorxy
-- End of GPT-2 Demo --
